In [1]:
# Import data handling libraries
import pandas as pd
import numpy as np

# Import text cleaning support
import re

# Import plotting libraries for quick checks if needed
import matplotlib.pyplot as plt
import seaborn as sns

# Import TF IDF vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Import baseline text classification models
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

# Import evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import model saving utility
import joblib

In [2]:
# Load sampled training data from csv
df_train = pd.read_csv(r"..\data\processed\cfpb_sample_500k.csv")

In [3]:
# Check dataset shape
df_train.shape

(500000, 4)

In [4]:
# Preview the first rows
df_train.head()

,text,target,text_clean,target_grouped
0,Hi I am submitting this XXXX XXXX this isn't a...,Credit reporting,hi i am submitting this xxxx xxxx this isn t a...,Credit reporting
1,A federal education loan was fraudulently open...,Debt collection,a federal education loan was fraudulently open...,Debt collection
2,I am writing to have the following information...,Credit reporting,i am writing to have the following information...,Credit reporting
3,I want this late payment remove from my accoun...,Credit card,i want this late payment remove from my accoun...,Credit card
4,I am disputing several items on my Experian cr...,Credit reporting,i am disputing several items on my experian cr...,Credit reporting


#### Bert

In [5]:
# Install transformer libraries for BERT fine tuning
!pip install transformers datasets accelerate


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Install the required packages in the notebook environment
!pip install -U "accelerate>=1.1.0" transformers datasets


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Import Hugging Face dataset utility
from datasets import Dataset

In [8]:
# Import tokenizer, model, trainer, and training arguments
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

#### Prepare BERT dataframe

In [9]:
# Create label mapping from grouped target
label2id = {label: i for i, label in enumerate(sorted(df_train["target_grouped"].unique()))}
id2label = {i: label for label, i in label2id.items()}

In [10]:
label2id

{'Auto loan': 0,
 'Bank account': 1,
 'Credit card': 2,
 'Credit reporting': 3,
 'Debt collection': 4,
 'Money transfer': 5,
 'Mortgage': 6,
 'Other': 7,
 'Personal loan': 8,
 'Student loan': 9}

In [11]:
id2label

{0: 'Auto loan',
 1: 'Bank account',
 2: 'Credit card',
 3: 'Credit reporting',
 4: 'Debt collection',
 5: 'Money transfer',
 6: 'Mortgage',
 7: 'Other',
 8: 'Personal loan',
 9: 'Student loan'}

In [12]:
# Create BERT-ready dataframe
df_bert = df_train[["text_clean", "target_grouped"]].copy()
df_bert["label"] = df_bert["target_grouped"].map(label2id)

In [13]:
df_bert

,text_clean,target_grouped,label
0,hi i am submitting this xxxx xxxx this isn t a...,Credit reporting,3
1,a federal education loan was fraudulently open...,Debt collection,4
2,i am writing to have the following information...,Credit reporting,3
3,i want this late payment remove from my accoun...,Credit card,2
4,i am disputing several items on my experian cr...,Credit reporting,3
...,...,...,...
499995,xxxx should not be reporting to xxxx xxxx and ...,Credit reporting,3
499996,the credit bureaus are reporting inaccurate ou...,Credit reporting,3
499997,we receive at least 3 calls an hour all day lo...,Debt collection,4
499998,these can be combined on my credit report you ...,Credit reporting,3


#### Sample smaller subset for BERT

In [14]:
# Sample a smaller subset for BERT fine tuning
df_bert_sample = df_bert.sample(n=50000, random_state=42)

In [15]:
train_df, test_df = train_test_split(
    df_bert_sample,
    test_size=0.2,
    random_state=42,
    stratify=df_bert_sample["label"]
)

In [16]:
# Check BERT train and test shapes
print(train_df.shape, test_df.shape)

(40000, 3) (10000, 3)


#### Convert to Hugging Face datasets

In [17]:
# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[["text_clean", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text_clean", "label"]])

#### Load Tokenizer

In [18]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

#### Tokenize text for BERT

In [19]:
# Tokenize the text for BERT
def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [20]:
# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [21]:
train_dataset

Dataset({
    features: ['text_clean', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 40000
})

In [22]:
test_dataset

Dataset({
    features: ['text_clean', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})

#### DistilBERT training

In [23]:
# Remove unused text columns and set dataset format for PyTorch
train_dataset = train_dataset.remove_columns(["text_clean", "__index_level_0__"])
test_dataset = test_dataset.remove_columns(["text_clean", "__index_level_0__"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [24]:
# Load DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
# Create metric function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    
    return {
        "accuracy": accuracy,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    }

In [26]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

In [27]:
# Create Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [28]:
# Create Trainer object again
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

#### Train the model

In [29]:
# Train DistilBERT model
trainer.train()

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1
1,0.502627,0.412235,0.881500,0.879289,0.668545
2,0.335748,0.427697,0.884300,0.881052,0.669472


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=10000, training_loss=0.4191873046875, metrics={'train_runtime': 43894.0929, 'train_samples_per_second': 1.823, 'train_steps_per_second': 0.228, 'total_flos': 5299451904000000.0, 'train_loss': 0.4191873046875, 'epoch': 2.0})

In [30]:
# Generate predictions on the test dataset
predictions = trainer.predict(test_dataset)

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [31]:
# Convert logits to predicted class ids
y_pred_bert = np.argmax(predictions.predictions, axis=1)

In [32]:
# Extract true labels from the test dataset
y_true_bert = predictions.label_ids

In [33]:
# Convert numeric predictions back to label names
y_pred_bert_labels = [id2label[i] for i in y_pred_bert]
y_true_bert_labels = [id2label[i] for i in y_true_bert]

In [34]:
# Print classification report for DistilBERT
print(classification_report(y_true_bert_labels, y_pred_bert_labels, zero_division=0))

                  precision    recall  f1-score   support

       Auto loan       0.58      0.60      0.59       127
    Bank account       0.81      0.78      0.79       491
     Credit card       0.74      0.74      0.74       608
Credit reporting       0.93      0.96      0.95      6682
 Debt collection       0.79      0.71      0.75      1121
  Money transfer       0.85      0.79      0.82       300
        Mortgage       0.87      0.84      0.85       380
           Other       0.00      0.00      0.00        14
   Personal loan       0.52      0.33      0.40       118
    Student loan       0.81      0.81      0.81       159

        accuracy                           0.88     10000
       macro avg       0.69      0.65      0.67     10000
    weighted avg       0.88      0.88      0.88     10000



In [35]:
# Calculate summary metrics for DistilBERT
bert_accuracy = accuracy_score(y_true_bert_labels, y_pred_bert_labels)
bert_weighted_f1 = f1_score(y_true_bert_labels, y_pred_bert_labels, average="weighted", zero_division=0)
bert_macro_f1 = f1_score(y_true_bert_labels, y_pred_bert_labels, average="macro", zero_division=0)

print("DistilBERT Accuracy:", bert_accuracy)
print("DistilBERT Weighted F1:", bert_weighted_f1)
print("DistilBERT Macro F1:", bert_macro_f1)

DistilBERT Accuracy: 0.8843
DistilBERT Weighted F1: 0.8810516129457635
DistilBERT Macro F1: 0.6694717200369779


In [36]:
# Save the fine-tuned DistilBERT model and tokenizer
trainer.save_model(r"../models/distilbert_complaint_classifier")
tokenizer.save_pretrained(r"../models/distilbert_complaint_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/distilbert_complaint_classifier\\tokenizer_config.json',
 '../models/distilbert_complaint_classifier\\tokenizer.json')

DistilBERT Result

DistilBERT achieved strong overall performance with 0.88 accuracy, 0.88 weighted F1, and 0.66 macro F1. This made it competitive with the strongest classical baseline, TF-IDF with Logistic Regression.

Interpretation

The BERT-based model improved class balance slightly relative to the standard TF-IDF baseline, as reflected in macro F1, but it did not clearly surpass the simpler model in overall weighted performance. This suggests that the complaint narratives contain strong keyword-level signal that is already well captured by TF-IDF, while transformer fine-tuning offers a more semantically robust but more computationally expensive alternative.

In [37]:
# Safely remove columns only if they exist
cols_to_remove = [col for col in ["text_clean", "__index_level_0__"] if col in train_dataset.column_names]
train_dataset = train_dataset.remove_columns(cols_to_remove)
test_dataset = test_dataset.remove_columns(cols_to_remove)

#### New Bert Sample

In [38]:
# Sample 200k rows for BERT fine tuning
df_bert_sample = df_bert.sample(n=150000, random_state=42)

In [39]:
# Check sampled BERT dataframe shape
df_bert_sample.shape

(150000, 3)

In [40]:
# Create train and test split for BERT
train_df, test_df = train_test_split(
    df_bert_sample,
    test_size=0.2,
    random_state=42,
    stratify=df_bert_sample["label"]
)

In [41]:
# Check train and test shapes
print(train_df.shape, test_df.shape)

(120000, 3) (30000, 3)


In [42]:
# Check grouped label distribution in the training split
train_df["target_grouped"].value_counts()

target_grouped
Credit reporting    80214
Debt collection     13262
Credit card          7295
Bank account         5996
Mortgage             4514
Money transfer       3712
Student loan         1927
Auto loan            1548
Personal loan        1370
Other                 162
Name: count, dtype: int64

In [43]:
# Check grouped label distribution in the test split
test_df["target_grouped"].value_counts()

target_grouped
Credit reporting    20053
Debt collection      3315
Credit card          1824
Bank account         1499
Mortgage             1129
Money transfer        928
Student loan          482
Auto loan             387
Personal loan         342
Other                  41
Name: count, dtype: int64

In [44]:
# Save the 200k BERT sample to processed data
#df_bert_sample.to_csv("data/processed/df_bert_sample_200k.csv", index=False)

#### Hugging face dataset

In [45]:
# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df[["text_clean", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text_clean", "label"]])

#### Tokenizer

In [46]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [47]:
# Tokenize the text for BERT
def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [48]:
# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

In [49]:
# Safely remove extra columns and set PyTorch format
cols_to_remove = [col for col in ["text_clean", "__index_level_0__"] if col in train_dataset.column_names]
train_dataset = train_dataset.remove_columns(cols_to_remove)
test_dataset = test_dataset.remove_columns(cols_to_remove)

train_dataset.set_format("torch")
test_dataset.set_format("torch")

BERT class weight/ train

In [50]:
# Import PyTorch and class weight utility
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight

In [51]:
# Get unique label ids from the training split
class_ids = sorted(train_df["label"].unique())

In [52]:
class_ids

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9)]

In [53]:
# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(class_ids),
    y=train_df["label"].values
)

In [54]:
# Convert class weights to a PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

In [55]:
# Check class weights
print(class_weights_tensor)

tensor([ 7.7519,  2.0013,  1.6450,  0.1496,  0.9048,  3.2328,  2.6584, 74.0741,
         8.7591,  6.2273])


#### Load DistilBERT

In [56]:
# Load DistilBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [57]:
# Create a custom Trainer that uses weighted cross-entropy loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(model.device))
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

> git push origin main:main
error: RPC failed; HTTP 408 curl 22 The requested URL returned error: 408
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date

In [58]:
# Create metric function for evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    
    return {
        "accuracy": accuracy,
        "weighted_f1": weighted_f1,
        "macro_f1": macro_f1
    }

In [59]:
# Define training arguments for weighted-loss DistilBERT
training_args = TrainingArguments(
    output_dir="./bert_results_weighted_250k",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

#### Trainer

In [60]:
# Create weighted Trainer object
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [61]:
# Train weighted DistilBERT model
trainer.train()

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Macro F1
1,0.869106,0.793421,0.889233,0.888941,0.688156
2,0.663086,0.807349,0.896300,0.895322,0.700030
3,0.538044,0.886221,0.896100,0.895459,0.700747


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=45000, training_loss=0.6900784722222222, metrics={'train_runtime': 194658.7374, 'train_samples_per_second': 1.849, 'train_steps_per_second': 0.231, 'total_flos': 2.3847533568e+16, 'train_loss': 0.6900784722222222, 'epoch': 3.0})

In [62]:
# Generate predictions on the test dataset
predictions = trainer.predict(test_dataset)

c:\Users\tevin\OneDrive\Desktop\cfpb-complaint-intelligence\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [63]:
# Convert logits to predicted class ids
y_pred_bert = np.argmax(predictions.predictions, axis=1)

In [64]:
# Extract true labels
y_true_bert = predictions.label_ids

In [65]:
# Convert numeric ids back to label names
y_pred_bert_labels = [id2label[i] for i in y_pred_bert]
y_true_bert_labels = [id2label[i] for i in y_true_bert]

In [66]:
# Print classification report
print(classification_report(y_true_bert_labels, y_pred_bert_labels, zero_division=0))

                  precision    recall  f1-score   support

       Auto loan       0.62      0.68      0.65       387
    Bank account       0.78      0.85      0.81      1499
     Credit card       0.78      0.77      0.78      1824
Credit reporting       0.95      0.95      0.95     20053
 Debt collection       0.78      0.78      0.78      3315
  Money transfer       0.86      0.78      0.82       928
        Mortgage       0.87      0.93      0.90      1129
           Other       0.00      0.00      0.00        41
   Personal loan       0.55      0.48      0.51       342
    Student loan       0.79      0.82      0.81       482

        accuracy                           0.90     30000
       macro avg       0.70      0.70      0.70     30000
    weighted avg       0.90      0.90      0.90     30000



In [67]:
# Calculate summary metrics for weighted DistilBERT
bert_accuracy = accuracy_score(y_true_bert_labels, y_pred_bert_labels)
bert_weighted_f1 = f1_score(y_true_bert_labels, y_pred_bert_labels, average="weighted", zero_division=0)
bert_macro_f1 = f1_score(y_true_bert_labels, y_pred_bert_labels, average="macro", zero_division=0)

print("Weighted DistilBERT Accuracy:", bert_accuracy)
print("Weighted DistilBERT Weighted F1:", bert_weighted_f1)
print("Weighted DistilBERT Macro F1:", bert_macro_f1)

Weighted DistilBERT Accuracy: 0.8961
Weighted DistilBERT Weighted F1: 0.8954588772088844
Weighted DistilBERT Macro F1: 0.7007465345354993


Bert shines more when the task needs:

- deeper context
- paraphrase understanding
- more subtle semantics

The task has some of that, but a lot of it is still keyword-driven.

DistilBERT is competitive, but this dataset is a great fit for TF-IDF + Logistic Regression